# HEST Feature Extraction (Colab)

Thin Colab wrapper around the pipeline script
`08a_extract_features.py`. The script does the actual work
(download HEST `.tif` from `MahmoodLab/hest`, tile in-memory, run
UNI, save `.h5` to Drive at `embeddings/uni_hest/{file_id}.h5`).
This notebook only handles Drive mount + HF login + invoking the
script with streamed output.

**Prerequisites**
- Hugging Face access to `MahmoodLab/hest` (gated; approved
  2026-05-14 for this project) and `MahmoodLab/uni`.
- Diagnostic manifest at
  `<Drive>/prame-predict/data/expression/diagnostic_manifest.csv`
  with at least one `source_group=='hest_visium'` row.

**Runtime** - L4 GPU, ~88 slides: roughly 25-40 min wall-clock
(HF download bandwidth is the bottleneck; GPU-resident extract is
~5 sec per slide).


In [ ]:
# Cell 1: Install dependencies, mount Drive, clone repo.
!pip install -q timm huggingface_hub openslide-python h5py opencv-python-headless
!apt-get install -qq -y openslide-tools

from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('prame-predict'):
    !git clone https://github.com/hb-1968/prame-predict.git
else:
    !cd prame-predict && git pull --ff-only


In [ ]:
# Cell 2: HuggingFace login (MahmoodLab/hest + MahmoodLab/uni are gated).
from huggingface_hub import login
login()


In [ ]:
# Cell 3: Paths.
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/prame-predict')
MANIFEST   = DRIVE_ROOT / 'data' / 'expression' / 'diagnostic_manifest.csv'
EMB_DIR    = DRIVE_ROOT / 'embeddings'           # 08a appends uni_hest/

LOCAL_REPO = Path('/content/prame-predict')

assert MANIFEST.exists(), (
    f'Manifest not found at {MANIFEST}. Run 08_build_diagnostic_manifest.py '
    'and sync data/expression/ to Drive first.'
)
print(f'Manifest: {MANIFEST}')
print(f'Embeddings dir (cohort subdir auto-appended): {EMB_DIR}')


In [ ]:
# Cell 4: Run 08a_extract_features.py with streaming output.
import os, select, subprocess, time

cmd = [
    'python', '-u', str(LOCAL_REPO / '08a_extract_features.py'),
    '--source-group', 'hest_visium',
    '--manifest', str(MANIFEST),
    '--emb-dir',  str(EMB_DIR),
    '--device',   'cuda',
    '--amp',
]
print('Command:')
print('  ' + ' '.join(cmd))
print()

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
t0 = time.time()
proc = subprocess.Popen(
    cmd, cwd=str(LOCAL_REPO),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env,
)

try:
    while True:
        while True:
            ready, _, _ = select.select([proc.stdout], [], [], 0.5)
            if not ready:
                break
            line = proc.stdout.readline()
            if not line:
                break
            print(line, end='', flush=True)
        if proc.poll() is not None:
            tail = proc.stdout.read()
            if tail:
                print(tail, end='', flush=True)
            break
finally:
    rc = proc.wait()

print(f'\nFinished in {(time.time() - t0) / 60:.1f} min  (exit code {rc})')


In [ ]:
# Cell 5: QC counts.
HEST_EMB_DIR = EMB_DIR / 'uni_hest'
on_drive = sorted(HEST_EMB_DIR.glob('*.h5')) if HEST_EMB_DIR.exists() else []
print(f'embeddings/uni_hest on Drive: {len(on_drive)} .h5 files')

import pandas as pd
df = pd.read_csv(MANIFEST)
total = int((df['source_group'] == 'hest_visium').sum())
print(f'manifest hest_visium rows:    {total}')
missing = total - len(on_drive)
if missing > 0:
    print(f'  [warn] {missing} hest_visium rows still missing on Drive.')
    print('         Inspect the failed list in the cell above and re-run if transient.')
else:
    print('  All hest_visium rows have embeddings.')
